In [12]:
import os, time, requests
import pandas as pd

HEADERS = {"User-Agent": "idlhaj ayoubcon2@gmail.com"}

def filing_url(row):
    acc = row["accessionNumber"].replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{row['cik']}/{acc}/{row['primaryDocument']}"

def local_path(row):
    ext = os.path.splitext(row["primaryDocument"])[1]
    return f"../data/raw/filings/{row['accessionNumber']}{ext}"

sample = pd.read_csv("../data/raw/filings_sample.csv")
sample = sample.dropna(subset=["primaryDocument"]).reset_index(drop=True)
print(len(sample))

1489


In [14]:
print(len(os.listdir("../data/raw/filings")))

1488


In [15]:
failed = []
for i, row in sample.iterrows():
    path = local_path(row)
    if os.path.exists(path):
        continue
    try:
        r = requests.get(filing_url(row), headers=HEADERS, timeout=30)
        if r.status_code == 200:
            with open(path, "wb") as f:
                f.write(r.content)
        else:
            failed.append((row["accessionNumber"], r.status_code))
    except Exception as e:
        failed.append((row["accessionNumber"], str(e)))
    time.sleep(0.15)

print("still failed:", len(failed))
print(failed)

still failed: 1
[('0001104659-00-000556', 404)]


In [6]:
import requests, random, time
import pandas as pd

HEADERS = {"User-Agent": "Your Name your.email@example.com"}   # put your real name and email

tickers = requests.get(
    "https://www.sec.gov/files/company_tickers.json", headers=HEADERS
).json()
print(len(tickers))

10434


In [7]:
random.seed(42)
sample = random.sample(list(tickers.values()), 500)
print(sample[0])

{'cik_str': 1969373, 'ticker': 'VTMX', 'title': 'Vesta Real Estate Corporation, S.A.B. de C.V.'}


In [ ]:
import os

path = "../data/raww/companies_meta_all.csv"

# unique CIKs (one company can appear several times in the list, e.g. share classes)
all_ciks = sorted({c["cik_str"] for c in tickers.values()})
print(len(all_ciks))

# if we already saved some, continue from there
if os.path.exists(path):
    saved = pd.read_csv(path)
    rows = saved.to_dict("records")
    done = set(saved["cik"])
else:
    rows, done = [], set()

for i, cik in enumerate(all_ciks):
    if cik in done:
        continue
    try:
        url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
        meta = requests.get(url, headers=HEADERS).json()
        rows.append({
            "cik": cik,
            "name": meta["name"],
            "sic": meta["sic"],
            "sicDescription": meta["sicDescription"],
        })
    except Exception as e:
        print("failed:", cik, e)
    time.sleep(0.15)
    if i % 500 == 0:
        pd.DataFrame(rows).to_csv(path, index=False)   # save progress

pd.DataFrame(rows).to_csv(path, index=False)
print("done:", len(rows))

8016
done: 8016


In [ ]:
def finance_sector(sic):
    if 6000 <= sic <= 6099:
        return "Banking"
    if 6100 <= sic <= 6199:
        return "Credit & Lending"
    if 6200 <= sic <= 6299:
        return "Securities & Brokerage"
    if 6300 <= sic <= 6499:
        return "Insurance"
    if 6500 <= sic <= 6599 or sic == 6798:
        return "Real Estate"
    if sic == 6770:
        return "Blank Check (SPAC)"
    return None

In [ ]:
df_all = pd.read_csv("../data/raww/companies_meta_all.csv")
df_all = df_all.dropna(subset=["sic"]).copy()      # remove empty sic
df_all["sic"] = df_all["sic"].astype(int)

df_fin = df_all[(df_all["sic"] >= 6000) & (df_all["sic"] <= 6799)].copy()
df_fin["sector"] = df_fin["sic"].apply(finance_sector)

print(len(df_fin))
print(df_fin["sector"].value_counts())

1686
sector
Banking                   405
Blank Check (SPAC)        349
Real Estate               299
Securities & Brokerage    226
Credit & Lending          220
Insurance                 152
Name: count, dtype: int64


In [ ]:
parts = []
for name, group in df_fin.groupby("sector"):
    parts.append(group.sample(n=100, random_state=42))

df_sel = pd.concat(parts)
print(len(df_sel))
print(df_sel["sector"].value_counts())

df_sel.to_csv("../data/raww/companies_selected.csv", index=False)

600
sector
Banking                   100
Blank Check (SPAC)        100
Credit & Lending          100
Insurance                 100
Real Estate               100
Securities & Brokerage    100
Name: count, dtype: int64


In [ ]:
cik = int(df_sel.iloc[0]["cik"])
meta = requests.get(
    f"https://data.sec.gov/submissions/CIK{cik:010d}.json", headers=HEADERS
).json()

recent = pd.DataFrame(meta["filings"]["recent"])
print(df_sel.iloc[0]["name"], "| sector:", df_sel.iloc[0]["sector"])
print("number of filings:", len(recent))
print(recent["form"].value_counts().head(15))
recent[["form", "filingDate", "accessionNumber", "primaryDocument"]].head()

COMMUNITY FINANCIAL SYSTEM, INC. | sector: Banking
number of filings: 1000
form
4           624
8-K         131
10-Q         34
SC 13G/A     28
4/A          27
3            21
425          16
144          13
11-K         11
DEFA14A      11
DEF 14A      11
10-K         11
SC 13G        8
S-8           7
5             6
Name: count, dtype: int64


,form,filingDate,accessionNumber,primaryDocument
0,4,2026-09-18,0000723188-26-000114,xslF345X06/form4.xml
1,4,2026-09-18,0000723188-26-000112,xslF345X06/form4.xml
2,4,2026-09-10,0000723188-26-000110,xslF345X06/form4.xml
3,4,2026-08-28,0000723188-26-000108,xslF345X06/form4.xml
4,4,2026-08-26,0000723188-26-000106,xslF345X06/form4.xml


In [ ]:
FORMS = ["10-K", "10-Q", "8-K", "DEF 14A", "S-1", "424B4"]

df_sel = pd.read_csv("../data/raww/companies_selected.csv")
all_filings = []

for _, row in df_sel.iterrows():
    cik = int(row["cik"])
    try:
        url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
        meta = requests.get(url, headers=HEADERS).json()
        recent = pd.DataFrame(meta["filings"]["recent"])
        recent = recent[recent["form"].isin(FORMS)]
        recent = recent[["form", "filingDate", "accessionNumber", "primaryDocument"]].copy()
        recent["cik"] = cik
        recent["sector"] = row["sector"]
        all_filings.append(recent)
    except Exception as e:
        print("failed:", cik, e)
    time.sleep(0.15)

filings = pd.concat(all_filings, ignore_index=True)
filings.to_csv("../data/raww/filings_index.csv", index=False)
print(len(filings))
print(pd.crosstab(filings["sector"], filings["form"]))

70336
form                    10-K  10-Q  424B4    8-K  DEF 14A  S-1
sector                                                        
Banking                  861  2652     14   9977      890   37
Blank Check (SPAC)       136   466     86    936       73  100
Credit & Lending         641  1674     78   7879      489  146
Insurance                853  2603     52   9519      835   51
Real Estate             1238  3806     53  14210     1137   15
Securities & Brokerage   699  2132     43   5475      339  141


In [ ]:
filings = pd.read_csv("../data/raww/filings_index.csv")

type_map = {
    "10-K": "Annual Report",
    "10-Q": "Quarterly Report",
    "8-K": "Event Report",
    "DEF 14A": "Proxy Statement",
    "S-1": "Prospectus",
    "424B4": "Prospectus",
}
filings["doc_type"] = filings["form"].map(type_map)

parts = []
for (sector, doc_type), group in filings.groupby(["sector", "doc_type"]):
    parts.append(group.sample(n=50, random_state=42))

sample = pd.concat(parts, ignore_index=True)
print(len(sample))
print(pd.crosstab(sample["sector"], sample["doc_type"]))

sample.to_csv("../data/raww/filings_sample.csv", index=False)

1500
doc_type                Annual Report  Event Report  Prospectus  \
sector                                                            
Banking                            50            50          50   
Blank Check (SPAC)                 50            50          50   
Credit & Lending                   50            50          50   
Insurance                          50            50          50   
Real Estate                        50            50          50   
Securities & Brokerage             50            50          50   

doc_type                Proxy Statement  Quarterly Report  
sector                                                     
Banking                              50                50  
Blank Check (SPAC)                   50                50  
Credit & Lending                     50                50  
Insurance                            50                50  
Real Estate                          50                50  
Securities & Brokerage               5

In [ ]:
import os

sample = pd.read_csv("../data/raww/filings_sample.csv")
os.makedirs("../data/raw/filings", exist_ok=True)

def filing_url(row):
    acc = row["accessionNumber"].replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{row['cik']}/{acc}/{row['primaryDocument']}"

test = sample.sample(5, random_state=1)
for _, row in test.iterrows():
    url = filing_url(row)
    r = requests.get(url, headers=HEADERS)
    print(row["doc_type"], "|", r.status_code, "|", len(r.content) // 1024, "KB |", url)
    time.sleep(0.15)

Event Report | 200 | 19 KB | https://www.sec.gov/Archives/edgar/data/1297341/000121390021005668/ea134253-8k_kentuckyfirst.htm
Event Report | 200 | 32 KB | https://www.sec.gov/Archives/edgar/data/828944/000162828022011935/wsfs-20220503.htm
Annual Report | 200 | 803 KB | https://www.sec.gov/Archives/edgar/data/1450923/000199937124002921/sgol-10k_123123.htm
Event Report | 200 | 9 KB | https://www.sec.gov/Archives/edgar/data/1098482/000095014404000013/g86554e8vk.htm
Event Report | 200 | 47 KB | https://www.sec.gov/Archives/edgar/data/1937926/000110465926058048/tm2613984d1_8k.htm


In [ ]:
print("missing:", sample["primaryDocument"].isna().sum())

sample = sample.dropna(subset=["primaryDocument"]).reset_index(drop=True)
print(len(sample))
print(pd.crosstab(sample["sector"], sample["doc_type"]))

missing: 11
1489
doc_type                Annual Report  Event Report  Prospectus  \
sector                                                            
Banking                            48            50          50   
Blank Check (SPAC)                 50            50          50   
Credit & Lending                   50            50          50   
Insurance                          50            50          50   
Real Estate                        50            49          50   
Securities & Brokerage             50            50          50   

doc_type                Proxy Statement  Quarterly Report  
sector                                                     
Banking                              47                50  
Blank Check (SPAC)                   50                50  
Credit & Lending                     49                50  
Insurance                            50                50  
Real Estate                          49                48  
Securities & Brokerage    

In [ ]:
def local_path(row):
    ext = os.path.splitext(row["primaryDocument"])[1]
    return f"../data/raw/filings/{row['accessionNumber']}{ext}"

failed = []
for i, row in sample.iterrows():
    path = local_path(row)
    if os.path.exists(path):
        continue
    try:
        r = requests.get(filing_url(row), headers=HEADERS, timeout=30)
        if r.status_code == 200:
            with open(path, "wb") as f:
                f.write(r.content)
        else:
            failed.append((row["accessionNumber"], r.status_code))
    except Exception as e:
        failed.append((row["accessionNumber"], str(e)))
    time.sleep(0.15)
    if i % 100 == 0:
        print(i, "done")

print("failed:", len(failed))

AttributeError: 'list' object has no attribute 'iterrows'

In [16]:
print(failed)
print(len(os.listdir("../data/raw/filings")))

[('0001104659-00-000556', 404)]
1488


In [17]:
sample["path"] = sample.apply(local_path, axis=1)
sample["downloaded"] = sample["path"].apply(os.path.exists)
print(sample["downloaded"].value_counts())

sample = sample[sample["downloaded"]].reset_index(drop=True)
sample.to_csv("../data/raw/filings_downloaded.csv", index=False)
print(len(sample))

downloaded
True     1488
False       1
Name: count, dtype: int64
1488


In [18]:
sample["size_kb"] = sample["path"].apply(lambda p: os.path.getsize(p) // 1024)
print(sample.groupby("doc_type")["size_kb"].describe()[["min", "50%", "max"]])

sample["year"] = pd.to_datetime(sample["filingDate"]).dt.year
print(sample["year"].value_counts().sort_index())

sample["ext"] = sample["path"].apply(lambda p: os.path.splitext(p)[1])
print(sample["ext"].value_counts())

                    min     50%      max
doc_type                                
Annual Report      12.0  2692.5  20728.0
Event Report        4.0    31.0    171.0
Prospectus        205.0  2300.5  24356.0
Proxy Statement    23.0   779.5   7140.0
Quarterly Report   55.0  1416.0  14170.0
year
2000      1
2001      5
2002      5
2003     11
2004      5
2005     10
2006     12
2007     14
2008     18
2009     21
2010     25
2011     30
2012     25
2013     34
2014     30
2015     41
2016     52
2017     71
2018     73
2019     72
2020     94
2021    116
2022    122
2023    125
2024    114
2025    179
2026    183
Name: count, dtype: int64
ext
.htm    1473
.txt      15
Name: count, dtype: int64


**DATA CLEANING**

In [25]:
pip install beautifulsoup4 lxml

   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.1 MB 1.2 MB/s eta 0:00:03
   ---------- ----------------------------- 1.0/4.1 MB 1.4 MB/s eta 0:00:03
   ---------- ----------------------------- 1.0/4.1 MB 1.4 MB/s eta 0:00:03
   ---------- ----------------------------- 1.0/4.1 MB 1.4 MB/s eta 0:00:03
   ------------ --------------------------- 1.3/4.1 MB 1.0 MB/s eta 0:00:03
   ------------------ --------------------- 1.8/4.1 MB 1.1 MB/s eta 0:00:02
   -------------------- ------------------- 2.1/4.1 MB 1.2 MB/s eta 0:00:02
   ------------------------- -------------- 2.6/4.1 MB 1.3 MB/s eta 0:00:02
   ------------------------------ --------- 3.1/4.1 MB 1.4 MB/s eta 0:00:01
   ------------------------------------ --- 3.7/4.1 MB 1.5 MB/s eta 0:00:01
   ------------------------------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
row = sample[sample["doc_type"] == "Event Report"].iloc[0]
with open(row["path"], "r", encoding="utf-8", errors="ignore") as f:
    raw = f.read()
print(len(raw))
print(raw[:1500])

27147
<DOCUMENT>
<TYPE>8-K
<SEQUENCE>1
<FILENAME>form8k_0910.htm
<DESCRIPTION>FORM 8K SEPTEMBER 30, 2010
<TEXT>
<html>
<head>
    <title>form8k_0910.htm</title>
    <!-- Licensed to: ccbg -->
    <!-- Document Created using EDGARizer 5.2.3.0 -->
    <!-- Copyright 1995 - 2009 Thomson Reuters. All rights reserved. -->
</head>





<body bgcolor="#ffffff" style="DISPLAY: inline; FONT-SIZE: 10pt; FONT-FAMILY: Times New Roman">
<br>
<br>
<div style="DISPLAY: block; MARGIN-LEFT: 0pt; TEXT-INDENT: 0pt; MARGIN-RIGHT: 0pt" align="center"><font style="DISPLAY: inline; FONT-WEIGHT: bold; FONT-SIZE: 10pt; FONT-FAMILY: Times New Roman">UNITED STATES</font></div>

<div style="DISPLAY: block; MARGIN-LEFT: 0pt; TEXT-INDENT: 0pt; MARGIN-RIGHT: 0pt" align="center"><font style="DISPLAY: inline; FONT-WEIGHT: bold; FONT-SIZE: 10pt; FONT-FAMILY: Times New Roman">SECURITIES AND EXCHANGE COMMISSION</font></div>

<div style="DISPLAY: block; MARGIN-LEFT: 0pt; TEXT-INDENT: 0pt; MARGIN-RIGHT: 0pt" align="center"

In [27]:
from bs4 import BeautifulSoup
import re

with open(row["path"], "r", encoding="utf-8", errors="ignore") as f:
    raw = f.read()

soup = BeautifulSoup(raw, "lxml")
for tag in soup(["script", "style"]):
    tag.decompose()

text = soup.get_text(separator=" ")
text = re.sub(r"\s+", " ", text).strip()

print(len(text))
print(text[:1500])

2737
8-K 1 form8k_0910.htm FORM 8K SEPTEMBER 30, 2010 form8k_0910.htm UNITED STATES SECURITIES AND EXCHANGE COMMISSION WASHINGTON, DC 20549 ____________________ FORM 8-K CURRENT REPORT Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934 Date of Report (Date of earliest event reported): October 26, 2010 CAPITAL CITY BANK GROUP, INC. (Exact name of registrant as specified in its charter) Florida 0-13358 59-2273542 (State of Incorporation) (Commission File Number) (IRS Employer Identification No.) 217 North Monroe Street, Tallahassee, Florida 32301 (Address of principal executive offices) (Zip Code) Registrant's telephone number, including area code: (850) 671-0300 ___________________________________________________ (Former Name or Former Address, if Changed Since Last Report) Check the appropriate box below if the Form 8-K filing is intended to simultaneously satisfy the filing obligation of the registrant under any of the following provisions (see General Instruction 

In [28]:
for word in ["8-K", "8K", "FORM 8", "10-K", "10-Q"]:
    print(word, text.count(word))

8-K 4
8K 1
FORM 8 3
10-K 0
10-Q 0


In [29]:
def mask_form_names(text):
    # technical leftovers: file names such as form8k_0910.htm
    text = re.sub(r"\S+\.htm\b", " ", text, flags=re.I)
    # form codes: 8-K, 8K, 10-K, 10-Q, DEF 14A, S-1, 424B4 (with optional /A)
    text = re.sub(r"\b(8-?K|10-?K|10-?Q|DEF\s*14A|S-1|424B\d)(/A)?\b", "[FORM]", text, flags=re.I)
    return re.sub(r"\s+", " ", text).strip()

masked = mask_form_names(text)
for word in ["8-K", "8K", "10-K", "10-Q", ".htm"]:
    print(word, masked.count(word))
print(masked[:600])

8-K 0
8K 0
10-K 0
10-Q 0
.htm 0
[FORM] 1 FORM [FORM] SEPTEMBER 30, 2010 UNITED STATES SECURITIES AND EXCHANGE COMMISSION WASHINGTON, DC 20549 ____________________ FORM [FORM] CURRENT REPORT Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934 Date of Report (Date of earliest event reported): October 26, 2010 CAPITAL CITY BANK GROUP, INC. (Exact name of registrant as specified in its charter) Florida 0-13358 59-2273542 (State of Incorporation) (Commission File Number) (IRS Employer Identification No.) 217 North Monroe Street, Tallahassee, Florida 32301 (Address of principal executive offices) (Zip Code) Regi


In [33]:
row2 = sample[sample["doc_type"] == "Quarterly Report"].iloc[0]

In [ ]:
with open(row2["path"], "r", encoding="utf-8", errors="ignore") as f:
    raw = f.read()

soup = BeautifulSoup(raw, "lxml")
for tag in soup(["script", "style"]):
    tag.decompose()

text = soup.get_text(separator=" ")
text = re.sub(r"\s+", " ", text).strip()

print(len(text))
print(text[:1500])

234961
10-Q 1 tv499039_10q.htm 10-Q UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-Q þ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For quarterly period ended June 30, 2018 ¨ TRANSITION REPORT UNDER SECTION 13 OR 15(d) OF THE EXCHANGE ACT For the transition period from _______________ to ________________ Commission file number 0-14237 First United Corporation (Exact name of registrant as specified in its charter) Maryland 52-1380770 (State or other jurisdiction of (I. R. S. Employer Identification No.) incorporation or organization) 19 South Second Street, Oakland, Maryland 21550-0009 (Address of principal executive offices) (Zip Code) (800) 470-4356 (Registrant's telephone number, including area code) Indicate by check mark whether the registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding 12 months (or for such shorter perio

In [40]:
masked2 = mask_form_names(text)
for word in ["10-Q", "10Q", "10-K", ".htm"]:
    print(word, masked2.count(word))
print(masked)

10-Q 0
10Q 0
10-K 0
.htm 0
[FORM] 1 FORM [FORM] SEPTEMBER 30, 2010 UNITED STATES SECURITIES AND EXCHANGE COMMISSION WASHINGTON, DC 20549 ____________________ FORM [FORM] CURRENT REPORT Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934 Date of Report (Date of earliest event reported): October 26, 2010 CAPITAL CITY BANK GROUP, INC. (Exact name of registrant as specified in its charter) Florida 0-13358 59-2273542 (State of Incorporation) (Commission File Number) (IRS Employer Identification No.) 217 North Monroe Street, Tallahassee, Florida 32301 (Address of principal executive offices) (Zip Code) Registrant's telephone number, including area code: (850) 671-0300 ___________________________________________________ (Former Name or Former Address, if Changed Since Last Report) Check the appropriate box below if the Form [FORM] filing is intended to simultaneously satisfy the filing obligation of the registrant under any of the following provisions (see General Instructi

In [41]:
import random

def make_excerpts(text, n=3, size=300, seed=42):
    words = text.split()
    chunks = [" ".join(words[i:i+size]) for i in range(0, len(words), size)]
    if len(chunks) > 5:
        chunks = chunks[1:]                       # skip the cover page
    def letter_ratio(c):
        return sum(ch.isalpha() for ch in c) / len(c)
    chunks = [c for c in chunks if len(c.split()) >= 100 and letter_ratio(c) > 0.7]
    random.seed(seed)
    return random.sample(chunks, min(n, len(chunks)))

excerpts = make_excerpts(masked2)
print(len(excerpts))
print(excerpts[0])

3
an alternative standard of creditworthiness consistent with Section 939A of the Dodd-Frank Act; (c) revisions to recognition of credit risk mitigation; (d) rules for risk weighting of equity exposures and past due loans, and (e) revised capital treatment for derivatives and repo-style transactions. Regulators may require higher capital ratios when warranted by the particular circumstances or risk profile of a given banking organization. In the current regulatory environment, banking organizations must stay well capitalized in order to receive favorable regulatory treatment on acquisition and other expansion activities and favorable risk-based deposit insurance assessments. Our capital policy establishes guidelines meeting these regulatory requirements and takes into consideration current or anticipated risks as well as potential future growth opportunities. 77 The following table presents our capital ratios: June 30, 2018 December 31, 2017 Required for Capital Adequacy Purposes Requi

In [42]:
sample = pd.read_csv("../data/raw/filings_downloaded.csv")
docs = sample[sample["path"].str.endswith(".htm")].reset_index(drop=True)
print(len(docs))

def html_to_text(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()
    soup = BeautifulSoup(raw, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator=" ")
    return re.sub(r"\s+", " ", text).strip()

rows = []
for i, r in docs.iterrows():
    try:
        text = mask_form_names(html_to_text(r["path"]))
        for k, ex in enumerate(make_excerpts(text, seed=i)):
            rows.append({
                "id": f"{r['accessionNumber']}_{k}",
                "text": ex,
                "sector": r["sector"],
                "doc_type": r["doc_type"],
                "cik": r["cik"],
                "form": r["form"],
                "filing_date": r["filingDate"],
                "accession": r["accessionNumber"],
            })
    except Exception as e:
        print("error:", r["accessionNumber"], e)
    if i % 100 == 0:
        print(i, "done")

dataset = pd.DataFrame(rows)
dataset.to_csv("../data/processed/dataset.csv", index=False)
print(len(dataset))
print(pd.crosstab(dataset["sector"], dataset["doc_type"]))

1473
0 done


C:\Users\HP\AppData\Local\Temp\ipykernel_22676\2649024606.py:8: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw, "lxml")


100 done
200 done
300 done
400 done
500 done
600 done
700 done
800 done
900 done
1000 done
1100 done
1200 done
1300 done
1400 done
4140
doc_type                Annual Report  Event Report  Prospectus  \
sector                                                            
Banking                           138            89         150   
Blank Check (SPAC)                150           119         150   
Credit & Lending                  144           111         150   
Insurance                         150            95         150   
Real Estate                       150            94         150   
Securities & Brokerage            150           108         150   

doc_type                Proxy Statement  Quarterly Report  
sector                                                     
Banking                             135               147  
Blank Check (SPAC)                  147               150  
Credit & Lending                    140               150  
Insurance                  

**data check**

In [43]:
dataset = pd.read_csv("../data/processed/dataset.csv")

# 1. size of the excerpts
dataset["n_words"] = dataset["text"].str.split().str.len()
print(dataset["n_words"].describe())

# 2. duplicated texts
print("duplicates:", dataset["text"].duplicated().sum())

# 3. companies per sector
print(dataset.groupby("sector")["cik"].nunique())

# 4. shortcut words: how often does each type name its own type?
keywords = {"Annual Report": "ANNUAL REPORT", "Quarterly Report": "QUARTERLY REPORT",
            "Event Report": "CURRENT REPORT", "Proxy Statement": "PROXY STATEMENT",
            "Prospectus": "PROSPECTUS"}
for doc_type, kw in keywords.items():
    sub = dataset[dataset["doc_type"] == doc_type]
    share = sub["text"].str.upper().str.contains(kw).mean()
    print(f"{doc_type}: {share:.0%} of excerpts contain '{kw}'")

count    4140.000000
mean      295.186232
std        24.744099
min       101.000000
25%       300.000000
50%       300.000000
75%       300.000000
max       300.000000
Name: n_words, dtype: float64
duplicates: 1
sector
Banking                   76
Blank Check (SPAC)        78
Credit & Lending          68
Insurance                 80
Real Estate               82
Securities & Brokerage    66
Name: cik, dtype: int64
Annual Report: 7% of excerpts contain 'ANNUAL REPORT'
Quarterly Report: 7% of excerpts contain 'QUARTERLY REPORT'
Event Report: 53% of excerpts contain 'CURRENT REPORT'
Proxy Statement: 27% of excerpts contain 'PROXY STATEMENT'
Prospectus: 20% of excerpts contain 'PROSPECTUS'


In [44]:
dataset = dataset.drop_duplicates(subset="text").reset_index(drop=True)

def mask_titles(text):
    pattern = r"\b(ANNUAL REPORT|QUARTERLY REPORT|CURRENT REPORT|PROXY STATEMENT|PROSPECTUS)\b"
    return re.sub(pattern, "[TITLE]", text, flags=re.I)

dataset["text_masked"] = dataset["text"].apply(mask_titles)
dataset = dataset.drop(columns=["n_words"])
dataset.to_csv("../data/processed/dataset.csv", index=False)
print(len(dataset))

for doc_type, kw in keywords.items():
    sub = dataset[dataset["doc_type"] == doc_type]
    share = sub["text_masked"].str.upper().str.contains(kw).mean()
    print(doc_type, f"{share:.0%}")

4139
Annual Report 1%
Quarterly Report 1%
Event Report 0%
Proxy Statement 1%
Prospectus 0%


**data preparation**

In [45]:
import numpy as np

companies = dataset[["cik", "sector"]].drop_duplicates()
rng = np.random.RandomState(42)
assign = {}

for sector, g in companies.groupby("sector"):
    ciks = g["cik"].to_numpy().copy()
    rng.shuffle(ciks)
    n = len(ciks)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    for c in ciks[:n_train]:
        assign[c] = "train"
    for c in ciks[n_train:n_train + n_val]:
        assign[c] = "val"
    for c in ciks[n_train + n_val:]:
        assign[c] = "test"

dataset["split"] = dataset["cik"].map(assign)
print(dataset["split"].value_counts())
print(pd.crosstab(dataset["split"], dataset["sector"]))
print(pd.crosstab(dataset["split"], dataset["doc_type"]))

split
train    2957
test      670
val       512
Name: count, dtype: int64
sector  Banking  Blank Check (SPAC)  Credit & Lending  Insurance  Real Estate  \
split                                                                           
test        102                 197               105        106           90   
train       501                 435               484        488          472   
val          56                  84               105        101          108   

sector  Securities & Brokerage  
split                           
test                        70  
train                      577  
val                         58  
doc_type  Annual Report  Event Report  Prospectus  Proxy Statement  \
split                                                                
test                132            94         111              177   
train               629           414         687              600   
val                 120           108         102               86   

doc_

In [46]:
os.makedirs("../data/splits", exist_ok=True)
for name in ["train", "val", "test"]:
    dataset[dataset["split"] == name].to_csv(f"../data/splits/{name}.csv", index=False)

train = dataset[dataset["split"] == "train"]
val = dataset[dataset["split"] == "val"]
test = dataset[dataset["split"] == "test"]
print(set(train["cik"]) & set(test["cik"]))
print(set(train["cik"]) & set(val["cik"]))
print(set(val["cik"]) & set(test["cik"]))

set()
set()
set()
